In [ ]:
### heuristictesting.ipynb is a modular and agnostic notebook for testing the M heuristic on different
### horizons.

import pandas as pd
import numpy as np

def m_heuristic_accuracy(df, margin_col='margin', price_open_col='value', price_close_col='value',
                         rows_per_M=12, M_start=1, M_end=10, M_step=0.25):
    """
    Calculate heuristic accuracy across horizons.
    
    Args:
        df : DataFrame with columns for margin, open, close.
        margin_col : str, column name for margin (B - S).
        price_open_col : str, column for horizon open price.
        price_close_col : str, column for horizon close price.
        rows_per_M : int, number of intake rows per M1 (default 12 for S5 -> M1).
        M_start, M_end, M_step : horizon range.

    Returns:
        DataFrame with horizon, rows, acc_pos, acc_neg, acc_avg.
    """
    results = []
    
    def horizon_accuracy(block):
        margin_block = block[margin_col].mean()
        price_change = block[price_close_col].iloc[-1] - block[price_open_col].iloc[0]
        return np.sign(margin_block), np.sign(price_change)

    for m in np.arange(M_start, M_end + M_step, M_step):
        block_size = int(rows_per_M * m)
        correct_pos = correct_neg = total_pos = total_neg = 0
        
        for i in range(0, len(df), block_size):
            block = df.iloc[i:i+block_size]
            if len(block) < block_size:
                continue
            m_sign, p_sign = horizon_accuracy(block)
            
            if p_sign > 0:
                total_pos += 1
                if m_sign == p_sign:
                    correct_pos += 1
            elif p_sign < 0:
                total_neg += 1
                if m_sign == p_sign:
                    correct_neg += 1

        acc_pos = correct_pos / total_pos if total_pos else np.nan
        acc_neg = correct_neg / total_neg if total_neg else np.nan
        acc_avg = (correct_pos + correct_neg) / (total_pos + total_neg) if (total_pos + total_neg) else np.nan
        results.append((round(m,2), block_size, acc_pos, acc_neg, acc_avg, total_pos, total_neg))
    
    return pd.DataFrame(results, columns=['M','Rows','Acc_Pos','Acc_Neg','Acc_Avg','Count_Pos','Count_Neg'])

In [5]:
# Recent pandas dataframe.

df = pd.read_csv("full_labeled_USD_JPY_2025-09-22_to_2025-09-25.csv")

print(m_heuristic_accuracy(df))

KeyError: 'Margin'